# Google Docs Auto-Upload for Scan Analysis

This notebook documents and exercises the automatic uploading of summary figures into the daily Google Docs scan log (issue #286).

Analyzers that opt in via `upload_to_gdoc: true` in their config will have their summary figure inserted into a 2×2 display table in the scan entry after each analysis completes.  See the [live_watch notebook](live_watch.ipynb) for the full live-running workflow.

## Enabling uploads in an analyzer config

Add two fields to any analyzer entry in your experiment YAML:

```yaml
analyzers:
  - type: array2d
    device_name: UC_GaiaMode
    priority: 0
    upload_to_gdoc: true
    gdoc_slot: 0        # 0 → row 1 col 0 | 1 → row 1 col 1
                        # 2 → row 2 col 0 | 3 → row 2 col 1
    image_analyzer:
      analyzer_class: image_analysis.offline_analyzers.beam_analyzer.BeamAnalyzer
      camera_config_name: UC_GaiaMode
```

## Persistent Drive folder (recommended)

By default images are uploaded to a shared staging folder that is periodically purged.  To store images permanently, add `ImageParentFolderID` to your experiment INI (e.g. `HTUparameters.ini`):

```ini
[DEFAULT]
LogID               = <today's doc id, updated by createExperimentLog>
ImageParentFolderID = <drive folder id for permanent image storage>
```

A per-day subfolder (`YYYY-MM-DD`) is created automatically inside that parent folder.

In [ ]:
import logging

from geecs_data_utils import ScanPaths, ScanTag
from geecs_data_utils.config_roots import image_analysis_config, scan_analysis_config

# Show only gdoc-related logs by default; increase verbosity as needed
for name in ("image_analysis", "scan_analysis", "geecs_data_utils"):
    logging.getLogger(name).setLevel(logging.ERROR)
logging.getLogger("scan_analysis.gdoc_upload").setLevel(logging.INFO)
logging.getLogger("scan_analysis.task_queue").setLevel(logging.INFO)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s \u2014 %(message)s",
)

image_analysis_config.set_base_dir(ScanPaths.paths_config.image_analysis_configs_path)
scan_analysis_config.set_base_dir(ScanPaths.paths_config.scan_analysis_configs_path)

## Mode 1 — Direct upload (fastest test)

Upload a single known image to a specific table cell.  Useful for verifying
that Drive upload and `insertImageToTableCell` work before running any analysis.

In [ ]:
from scan_analysis.gdoc_upload import upload_summary_to_gdoc

# ── edit these ──────────────────────────────────────────────────────────────────────────
scan_tag = ScanTag(year=2025, month=4, day=3, number=2, experiment="Undulator")

image_path = "/path/to/existing/summary_figure.png"  # any PNG on disk

# Paste from the Google Doc URL: https://docs.google.com/document/d/<DOC_ID>/edit
# Leave as None to use the ID stored in HTUparameters.ini
document_id = None

gdoc_slot = 0  # 0–3
# ────────────────────────────────────────────────────────────────────────────

ok = upload_summary_to_gdoc(
    scan_tag=scan_tag,
    display_files=[image_path],
    gdoc_slot=gdoc_slot,
    document_id=document_id,
)
print("Upload succeeded" if ok else "Upload failed or skipped (check logs)")

## Mode 2 — Upload from existing analysis status files

After analysis has already run for a scan, `display_files` are stored in
`<scan_folder>/analysis_status/<analyzer_id>.yaml`.  These cells re-read those
records and upload without re-running any analysis.

> **Note:** `display_files` are only populated in status YAMLs for analyses run
> after this PR.  For older runs use Mode 1 with a known image path instead.

In [ ]:
from scan_analysis.task_queue import STATUS_DIR_NAME, TaskStatus

# ── edit these ──────────────────────────────────────────────────────────────────────────
scan_tag = ScanTag(year=2025, month=4, day=3, number=2, experiment="Undulator")
document_id = None  # or paste a specific doc ID string
# ────────────────────────────────────────────────────────────────────────────

scan_folder = ScanPaths.get_scan_folder_path(tag=scan_tag)
status_dir = scan_folder / STATUS_DIR_NAME

statuses = [TaskStatus.from_file(f) for f in sorted(status_dir.glob("*.yaml"))]

print(f"Found {len(statuses)} status file(s) in {status_dir}\n")
for st in statuses:
    print(f"  {st.analyzer_id:40s}  state={st.state}")
    if st.display_files:
        for f in st.display_files:
            print(f"    {f}")

In [ ]:
from scan_analysis.gdoc_upload import upload_summary_to_gdoc

# Map analyzer_id → gdoc_slot; only listed analyzers are uploaded
upload_plan = {
    "UC_GaiaMode": 0,
    "U_BCaveICT": 1,
}

status_map = {st.analyzer_id: st for st in statuses}

for analyzer_id, slot in upload_plan.items():
    st = status_map.get(analyzer_id)
    if st is None:
        print(f"{analyzer_id}: no status file found, skipping.")
        continue
    if not st.display_files:
        print(f"{analyzer_id}: no display_files recorded, skipping.")
        continue
    print(f"{analyzer_id} → slot {slot}: {st.display_files[-1]}")
    ok = upload_summary_to_gdoc(
        scan_tag=scan_tag,
        display_files=st.display_files,
        gdoc_slot=slot,
        document_id=document_id,
    )
    print(f"  {'succeeded' if ok else 'failed or skipped'}")

## Mode 3 — Full pipeline replay on a historical date

Uses `LiveTaskRunner` against a past date exactly as it would run live.
`rerun_completed=True` resets any previously-done analyses back to queued so
they execute again, and `display_files` are written to the status YAMLs as
they complete.

> **Note on the document ID:** `insertImageToExperimentLog` reads `LogID`
> from the experiment INI, which normally holds *today's* doc ID.  For a true
> historical backtest either:
> - temporarily set `LogID` in `HTUparameters.ini` to the old doc's ID, or
> - disable `upload_to_gdoc` in the config during the replay and use Mode 2
>   above to test the upload step separately.

In [ ]:
import time

from scan_analysis.live_task_runner import LiveTaskRunner

logging.getLogger("scan_analysis.live_task_runner").setLevel(logging.INFO)

# ── edit these ──────────────────────────────────────────────────────────────────────────
date_tag = ScanTag(year=2025, month=4, day=3, number=0, experiment="Undulator")
analyzer_group = "HTU"  # name of the experiment config to load
# ────────────────────────────────────────────────────────────────────────────

runner = LiveTaskRunner(
    analyzer_group=analyzer_group,
    date_tag=date_tag,
    config_dir=scan_analysis_config.base_dir,
    image_config_dir=image_analysis_config.base_dir,
)

runner.start()
try:
    while True:
        runner.process_new(
            base_directory=ScanPaths.paths_config.base_path,
            max_items=1,
            dry_run=False,
            rerun_completed=True,  # re-run analyses already marked done
            rerun_failed=True,
        )
        time.sleep(1)
finally:
    runner.stop()